<a href="https://colab.research.google.com/github/ValentinaZubareva2906/make_AI_product/blob/main/chain/memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-openai openai -q

## Если используете ключ из курса, запустите эти ячейки 👇

In [2]:
from langchain_openai import ChatOpenAI
from getpass import getpass

course_api_key= "sk-CTWDfcT-MqN2gUhZh_3qbA"

# инициализируем языковую модель
llm = ChatOpenAI(api_key=course_api_key, model='gpt-4o-mini',
                 base_url="https://aleron-llm.neuraldeep.tech/")

In [3]:
!pip install langchain langchain-classic openai langchain-openai langchain-community -U -q
from langchain_classic import PromptTemplate

💾 Добавим памяти LLM'ке! 🦙







In [4]:
# подгрузим датасет
import pandas as pd

df = pd.read_csv("https://stepik.org/media/attachments/lesson/1084404/dial_df.csv")
df.head()

,dialogue_id,question
0,1,Под каким номером играл Криштиану Роналду в Юв...
1,1,Под каким номером он играл в Манчестер Юнайтед?
2,1,В каком году он был рожден?
3,2,Сколько лет прожил величайший русский поэт Але...
4,2,В каком году он родился?


In [5]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Создадим словарь, который будет мапить session_id с историей диалога этой сессии
store = {}

# Напишем функцию, которая возвращает историю диалога по session ID.
def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


In [6]:
#создание промпт для диалога
template = """Ты ассистент, который отвечает на вопросы.
Отвечай **только одним целым числом (тип int)**, без пояснений и лишних символов.

История диалога:
{history}

Текущий вопрос:
{text}
"""

In [7]:
prompt = PromptTemplate(input_variables=["text", "history"], template=template)

In [8]:
chain = prompt | llm

In [9]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="text",  # указываем название переменой запроса
    history_messages_key="history", # название переменной для истории из шаблона
)

In [10]:
ans = []
for id, q in zip(df['dialogue_id'], df['question']):
  try:
    session_id = str(id)
    response = chain_with_history.invoke(
        {"text": q},
        config={"configurable": {"session_id": session_id}}
        )
    answer_text = response.content.strip()

    # Формируем запись для результата
    #result = {answer_text}

    ans.append(answer_text)

  except Exception as e:
    ans.append(-1)

In [11]:
df['answer'] = [int(x) for x in ans]
df

,dialogue_id,question,answer
0,1,Под каким номером играл Криштиану Роналду в Юв...,7
1,1,Под каким номером он играл в Манчестер Юнайтед?,7
2,1,В каком году он был рожден?,1985
3,2,Сколько лет прожил величайший русский поэт Але...,37
4,2,В каком году он родился?,1799
5,2,Дуэль в каком году стала для него последней?,1837
6,3,Сколько было участников в золотом составе груп...,4
7,3,В каком году родился барабанщик золотого соста...,-1
8,3,В каком году распалась группа?,1970
9,4,В каком году родился Альберт Эйнштейн?,1879


In [13]:
#df['answer'] = ans # запишем предсказания в датафрейм
df.to_csv('solution_3.csv', index=False) # сохраним решение в csv